[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_08/notebook_8_7_hallucination_detection.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 8.7: Hallucination Detection in Clinical LLMs

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand types of LLM hallucinations in clinical contexts
2. Implement consistency checking across multiple generations
3. Build fact verification systems using knowledge bases
4. Detect hallucinations through self-consistency and entailment
5. Elicit stated confidence (uncertainty elicitation) and test how well it predicts accuracy (calibration)
6. Implement production-ready hallucination detection pipelines

## Clinical Context

**The Challenge**: LLM hallucinations in healthcare are dangerous:
- LLMs may generate plausible but incorrect medical information
- Even the best models show roughly 5–15% hallucination rates on medical tasks (Chapter 8); rates vary widely by model, task and prompt
- Hallucinations can include incorrect dosages, contraindications, or diagnoses
- High confidence in incorrect information is particularly risky

**The Solution**: Multi-layered hallucination detection:
- Consistency checking: Generate multiple responses, check for agreement
- Fact verification: Compare claims against trusted knowledge bases
- Self-verification: Ask LLM to critique its own outputs
- Confidence calibration: Detect overconfident incorrect responses

**What the book emphasises**:
- Each detection method has blind spots: consistency checks miss *consistent* hallucinations; knowledge-base checks only cover claims with authoritative sources (Chapter 8)
- "Defense in depth — combining multiple methods — reduces but does not eliminate risk" (Chapter 8)
- Human oversight remains essential for high-stakes clinical applications

## Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional, Set
import re
from dataclasses import dataclass
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# For text similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")
print("\nNote: This notebook demonstrates hallucination detection techniques.")

Libraries imported successfully!

Note: This notebook demonstrates hallucination detection techniques.


## 🔑 Optional: Run This Notebook with a Real LLM

**Everything in this notebook runs without an API key.** By default it uses built-in *simulated* LLM responses, so results are free, fast, and reproducible. If you want to compare them with a real model, follow these steps. Cells marked **🔴 LIVE** then call the real model; without a key they just print a note and skip.

**1. Get a key (about 5 minutes).** We use [OpenRouter](https://openrouter.ai/): one key gives you models from Anthropic, OpenAI, Google, Meta and others through one OpenAI-compatible API.
1. Sign up at https://openrouter.ai/ (Google, GitHub or email).
2. Open **Keys** (https://openrouter.ai/settings/keys) → **Create Key**. Copy it now (it starts with `sk-or-v1-`; it is shown only once).
3. Add a small amount of credit (a few dollars is plenty for all of Chapter 8), **or** pick a free model, i.e. one whose ID ends in `:free` on https://openrouter.ai/models. Tip: set a credit limit on the key.

**2. Give the key to the notebook. Never paste it into a cell you might share or commit.**
- **Google Colab:** click 🔑 **Secrets** in the left sidebar → *Add new secret* → name `OPENROUTER_API_KEY`, paste the key, switch on *Notebook access*.
- **Local Jupyter / VS Code:** in a terminal, `export OPENROUTER_API_KEY="sk-or-v1-..."` (Windows: `setx OPENROUTER_API_KEY "sk-or-v1-..."`), then start Jupyter from that terminal.
- **Any environment, this session only:** run `import os, getpass; os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter key: ")` in a new cell.

**3. Re-run the setup cell below.** It should print **✓ Live LLM enabled**.

**Using a provider directly instead?** Use the same code with a different `base_url`, key and model: OpenAI (`base_url=None`, key from https://platform.openai.com/api-keys) or Anthropic (`base_url="https://api.anthropic.com/v1/"`, key from https://console.anthropic.com/). Check each provider's model list for current model IDs.

> ⚠️ **Privacy:** never send real patient data to a commercial API unless your institution has a data-processing agreement (e.g., a BAA in the US or a DPA under GDPR). All cases in this notebook are synthetic.
>
> 💡 Live calls need Colab or local Jupyter; they do not work in the in-browser (JupyterLite) version.
> 💰 Small models cost a fraction of a cent per call. Check the current price on the model's OpenRouter page.

In [2]:
# Optional live-LLM setup: safe to run without a key (the notebook then uses simulated responses)
import os

LLM_MODEL = "~anthropic/claude-haiku-latest"  # any model ID from https://openrouter.ai/models

def _find_openrouter_key():
    key = os.environ.get("OPENROUTER_API_KEY")
    if not key:
        try:
            from google.colab import userdata  # Colab 🔑 Secrets panel
            key = userdata.get("OPENROUTER_API_KEY")
        except Exception:
            pass
    return key

llm_client = None
_key = _find_openrouter_key()
if _key:
    try:
        from openai import OpenAI  # OpenRouter speaks the OpenAI API: pip install "openai>=1.0"
        llm_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=_key)
        print(f"✓ Live LLM enabled: {LLM_MODEL}  (calls use your OpenRouter credit)")
    except ImportError:
        print('Key found, but the openai package is missing. Run  %pip install "openai>=1.0"  and re-run this cell.')
else:
    print("ℹ️  No OPENROUTER_API_KEY found: using simulated LLM responses (free).")
    print("   To use a real model, follow the steps in the markdown cell above.")

def ask_llm(prompt, system="You are a careful clinical assistant.", temperature=0.0, max_tokens=600):
    """Send one prompt to the live model. Returns the reply text, or None when no key is configured."""
    if llm_client is None:
        return None
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

NO_KEY_MSG = "⏭️  Skipped: no API key configured (see '🔑 Optional: Run This Notebook with a Real LLM'). The simulated output above is what this cell would compare against."

ℹ️  No OPENROUTER_API_KEY found: using simulated LLM responses (free).
   To use a real model, follow the steps in the markdown cell above.


## Part 1: Understanding Clinical LLM Hallucinations

### Types of Hallucinations

**1. Factual Hallucinations**
- Incorrect drug dosages ("Aspirin 1000mg daily" instead of 100mg)
- Wrong medication names ("Metformin for hypertension")
- Fabricated lab values or vital signs

**2. Temporal Hallucinations**
- Incorrect timelines ("Stop clopidogrel 5 weeks before surgery" instead of 5 days)
- Wrong follow-up intervals

**3. Logical Inconsistencies**
- Self-contradictory statements
- Conflicting recommendations

**4. Source Fabrication**
- Citing non-existent guidelines
- Attributing false information to real sources

### Detection Strategies

1. **Self-Consistency**: Generate multiple responses, check for agreement
2. **Fact Verification**: Compare against trusted knowledge bases
3. **Entailment Checking**: Verify logical consistency
4. **Confidence Calibration**: Detect overconfident errors
5. **Multi-Model Consensus**: Use multiple LLMs, flag disagreements

In [3]:
@dataclass
class LLMResponse:
    """Represents an LLM response to be checked for hallucinations."""
    response_id: str
    query: str
    response_text: str
    metadata: Dict

@dataclass
class HallucinationDetectionResult:
    """Results from hallucination detection."""
    response_id: str
    is_hallucination: bool
    confidence: float
    hallucination_type: Optional[str]
    evidence: Dict

@dataclass
class ClinicalFact:
    """Represents a verified clinical fact."""
    fact_id: str
    fact_text: str
    category: str
    source: str
    metadata: Dict

print("Hallucination detection data structures defined!")

Hallucination detection data structures defined!


## Part 2: Self-Consistency Checking

### Method: Generate Multiple Responses and Check Agreement

In [4]:
class SelfConsistencyChecker:
    """
    Detects hallucinations by generating multiple responses and checking consistency.
    Inconsistent responses across generations suggest hallucination.
    """

    def __init__(self, num_samples: int = 5, consistency_threshold: float = 0.6):
        """
        Args:
            num_samples: Number of responses to generate
            consistency_threshold: Minimum similarity for consistency (0-1)
        """
        self.num_samples = num_samples
        self.consistency_threshold = consistency_threshold

    def _extract_key_claims(self, text: str) -> List[str]:
        """Extract key factual claims from response."""
        # Split into sentences
        sentences = re.split(r'(?<!\d)[.!?]+(?!\d)', text)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

        # Filter for medical claims (simplified)
        medical_keywords = ['mg', 'dose', 'treatment', 'medication', 'diagnosis',
                           'recommend', 'contraindicated', 'monitor']

        claims = []
        for sentence in sentences:
            if any(keyword in sentence.lower() for keyword in medical_keywords):
                claims.append(sentence)

        return claims

    def _compute_pairwise_similarity(self, responses: List[str]) -> np.ndarray:
        """Compute pairwise similarity matrix for responses."""
        if len(responses) < 2:
            return np.array([[1.0]])

        # Use TF-IDF + cosine similarity
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(responses)
        similarity_matrix = cosine_similarity(tfidf_matrix)

        return similarity_matrix

    def check_consistency(self, responses: List[LLMResponse]) -> HallucinationDetectionResult:
        """Check consistency across multiple responses."""
        if len(responses) < 2:
            return HallucinationDetectionResult(
                response_id=responses[0].response_id if responses else "N/A",
                is_hallucination=False,
                confidence=0.0,
                hallucination_type=None,
                evidence={'error': 'Insufficient responses for consistency check'}
            )

        # Extract response texts
        response_texts = [r.response_text for r in responses]

        # Compute similarity matrix
        similarity_matrix = self._compute_pairwise_similarity(response_texts)

        # Calculate average pairwise similarity (excluding diagonal)
        n = len(responses)
        avg_similarity = (similarity_matrix.sum() - n) / (n * (n - 1)) if n > 1 else 1.0

        # Check for inconsistent claims
        all_claims = []
        for response in responses:
            claims = self._extract_key_claims(response.response_text)
            all_claims.extend(claims)

        # Determine if hallucination based on consistency
        is_hallucination = avg_similarity < self.consistency_threshold

        return HallucinationDetectionResult(
            response_id=responses[0].response_id,
            is_hallucination=is_hallucination,
            confidence=1.0 - avg_similarity if is_hallucination else avg_similarity,
            hallucination_type='inconsistency' if is_hallucination else None,
            evidence={
                'avg_similarity': float(avg_similarity),
                'threshold': self.consistency_threshold,
                'num_responses': len(responses),
                'similarity_matrix': similarity_matrix.tolist()
            }
        )

# Test with synthetic responses
test_responses_consistent = [
    LLMResponse(
        response_id="R1",
        query="What is the first-line treatment for type 2 diabetes?",
        response_text="Metformin is the first-line medication for type 2 diabetes. The starting dose is typically 500mg once or twice daily with meals. Titrate gradually to minimize GI side effects.",
        metadata={}
    ),
    LLMResponse(
        response_id="R2",
        query="What is the first-line treatment for type 2 diabetes?",
        response_text="The first-line pharmacologic treatment for type 2 diabetes is metformin. Initial dosing is 500mg daily, increased weekly to target of 2000mg daily in divided doses.",
        metadata={}
    ),
    LLMResponse(
        response_id="R3",
        query="What is the first-line treatment for type 2 diabetes?",
        response_text="Metformin should be used as initial therapy for type 2 diabetes. Start at 500mg once daily and titrate to 1000-2000mg daily based on tolerance.",
        metadata={}
    )
]

test_responses_inconsistent = [
    LLMResponse(
        response_id="R4",
        query="What is the first-line treatment for type 2 diabetes?",
        response_text="Metformin 500mg daily is the first-line treatment for type 2 diabetes.",
        metadata={}
    ),
    LLMResponse(
        response_id="R5",
        query="What is the first-line treatment for type 2 diabetes?",
        response_text="Insulin therapy should be started immediately for all type 2 diabetes patients at 10 units daily.",
        metadata={}
    ),
    LLMResponse(
        response_id="R6",
        query="What is the first-line treatment for type 2 diabetes?",
        response_text="Glipizide 5mg twice daily is the preferred initial treatment for type 2 diabetes mellitus.",
        metadata={}
    )
]

# Test consistency checker
consistency_checker = SelfConsistencyChecker(consistency_threshold=0.6)

print("Self-Consistency Check Results:\n")

result_consistent = consistency_checker.check_consistency(test_responses_consistent)
print("Consistent Responses:")
print(f"  Hallucination Detected: {result_consistent.is_hallucination}")
print(f"  Average Similarity: {result_consistent.evidence['avg_similarity']:.3f}")
print(f"  Confidence: {result_consistent.confidence:.3f}")

print("\nInconsistent Responses:")
result_inconsistent = consistency_checker.check_consistency(test_responses_inconsistent)
print(f"  Hallucination Detected: {result_inconsistent.is_hallucination}")
print(f"  Average Similarity: {result_inconsistent.evidence['avg_similarity']:.3f}")
print(f"  Confidence: {result_inconsistent.confidence:.3f}")

# Did the checker separate the two sets? (computed)
sim_c = result_consistent.evidence['avg_similarity']
sim_i = result_inconsistent.evidence['avg_similarity']
if result_consistent.is_hallucination:
    print(f"\n⚠️  FALSE ALARM: the consistent set is also flagged (similarity {sim_c:.3f} < threshold {consistency_checker.consistency_threshold}).")
    print("   All three agree on metformin 500mg, but they are paraphrases, so TF-IDF word overlap is low.")
    print(f"   The two sets differ by only {sim_c - sim_i:.3f} in similarity — word overlap is a weak proxy for agreement on facts.")
    print("   Exercise: compare the extracted drug + dose across responses instead of raw words, or calibrate the threshold on labelled data.")
elif result_inconsistent.is_hallucination:
    print("\n✓ The checker separates the two sets at this threshold.")
else:
    print("\n⚠️  MISSED: the inconsistent set was not flagged at this threshold.")

Self-Consistency Check Results:

Consistent Responses:
  Hallucination Detected: True
  Average Similarity: 0.272
  Confidence: 0.728

Inconsistent Responses:
  Hallucination Detected: True
  Average Similarity: 0.223
  Confidence: 0.777

⚠️  FALSE ALARM: the consistent set is also flagged (similarity 0.272 < threshold 0.6).
   All three agree on metformin 500mg, but they are paraphrases, so TF-IDF word overlap is low.
   The two sets differ by only 0.049 in similarity — word overlap is a weak proxy for agreement on facts.
   Exercise: compare the extracted drug + dose across responses instead of raw words, or calibrate the threshold on labelled data.


### 🔴 LIVE: Self-Consistency on a Real LLM

We sample the same question 5 times at a higher temperature and score agreement with the `consistency_checker` defined above. **Predict first:** will a real model be more or less consistent than the simulated responses?

In [5]:
# 🔴 LIVE: sample a real model several times and measure consistency
if llm_client is None:
    print(NO_KEY_MSG)
else:
    question = "What is the recommended starting dose of metformin for an adult with newly diagnosed type 2 diabetes, and when should it be avoided?"
    samples = [LLMResponse(response_id=f"live_{i}", query=question,
                           response_text=ask_llm(question, temperature=0.9, max_tokens=200),
                           metadata={"model": LLM_MODEL})
               for i in range(5)]
    for r in samples:
        print(f"[{r.response_id}] {r.response_text[:300]}\n")
    result = consistency_checker.check_consistency(samples)
    print(f"Average pairwise similarity: {result.evidence['avg_similarity']:.2f}  (threshold {result.evidence['threshold']})")
    print("Flagged as inconsistent" if result.is_hallucination else "Consistent across samples")

⏭️  Skipped: no API key configured (see '🔑 Optional: Run This Notebook with a Real LLM'). The simulated output above is what this cell would compare against.


## Part 3: Fact Verification Against Knowledge Base

### Method: Compare Claims Against Verified Clinical Facts

In [6]:
class FactVerificationChecker:
    """
    Verifies LLM claims against a knowledge base of verified clinical facts.
    """

    def __init__(self, knowledge_base: List[ClinicalFact]):
        self.knowledge_base = knowledge_base

        # Build fact index for efficient search
        self.fact_texts = [fact.fact_text for fact in knowledge_base]
        if self.fact_texts:
            self.vectorizer = TfidfVectorizer()
            self.fact_vectors = self.vectorizer.fit_transform(self.fact_texts)

    def _extract_numerical_claims(self, text: str) -> List[Dict]:
        """Extract claims with numerical values (dosages, lab values)."""
        # Pattern for dosages: number + unit
        dosage_pattern = r'(\d+(?:\.\d+)?)\s*(mg|g|ml|mcg|units?)'

        matches = re.finditer(dosage_pattern, text, re.IGNORECASE)
        claims = []
        for match in matches:
            # Extract context (surrounding text)
            start = max(0, match.start() - 50)
            end = min(len(text), match.end() + 50)
            context = text[start:end]

            claims.append({
                'value': float(match.group(1)),
                'unit': match.group(2).lower(),
                'context': context,
                'full_match': match.group(0)
            })

        return claims

    def _find_similar_facts(self, claim: str, top_k: int = 3) -> List[Tuple[ClinicalFact, float]]:
        """Find most similar facts from knowledge base."""
        if not self.fact_texts:
            return []

        # Vectorize claim
        claim_vector = self.vectorizer.transform([claim])

        # Compute similarities
        similarities = cosine_similarity(claim_vector, self.fact_vectors)[0]

        # Get top-k
        top_indices = similarities.argsort()[-top_k:][::-1]

        results = [(self.knowledge_base[i], float(similarities[i])) for i in top_indices]
        return results

    def verify_response(self, response: LLMResponse,
                       similarity_threshold: float = 0.5) -> HallucinationDetectionResult:
        """Verify response against knowledge base."""
        # Extract numerical claims
        numerical_claims = self._extract_numerical_claims(response.response_text)

        # Find similar facts for each claim
        verifications = []
        for claim_dict in numerical_claims:
            similar_facts = self._find_similar_facts(claim_dict['context'], top_k=3)

            if similar_facts:
                best_match, similarity = similar_facts[0]
                is_verified = similarity >= similarity_threshold

                verifications.append({
                    'claim': claim_dict['full_match'],
                    'context': claim_dict['context'],
                    'verified': is_verified,
                    'similarity': similarity,
                    'matched_fact': best_match.fact_text if is_verified else None
                })

        # Determine if hallucination
        if not verifications:
            # No numerical claims to verify
            is_hallucination = False
            confidence = 0.0
        else:
            verified_count = sum(1 for v in verifications if v['verified'])
            verification_rate = verified_count / len(verifications)

            # Hallucination if < 50% of claims verified
            is_hallucination = verification_rate < 0.5
            confidence = 1.0 - verification_rate if is_hallucination else verification_rate

        return HallucinationDetectionResult(
            response_id=response.response_id,
            is_hallucination=is_hallucination,
            confidence=confidence,
            hallucination_type='unverified_fact' if is_hallucination else None,
            evidence={
                'num_claims': len(numerical_claims),
                'verifications': verifications,
                'verification_rate': verification_rate if verifications else None
            }
        )

# Create synthetic knowledge base
knowledge_base = [
    ClinicalFact(
        fact_id="F001",
        fact_text="Metformin starting dose is 500mg once or twice daily with meals.",
        category="medication_dosing",
        source="ADA Guidelines 2024",
        metadata={}
    ),
    ClinicalFact(
        fact_id="F002",
        fact_text="Metformin target dose is 2000mg daily in divided doses.",
        category="medication_dosing",
        source="ADA Guidelines 2024",
        metadata={}
    ),
    ClinicalFact(
        fact_id="F003",
        fact_text="Aspirin for cardiovascular prevention is 81-100mg daily.",
        category="medication_dosing",
        source="AHA/ACC Guidelines 2023",
        metadata={}
    ),
    ClinicalFact(
        fact_id="F004",
        fact_text="Lisinopril starting dose for hypertension is 10mg daily.",
        category="medication_dosing",
        source="ESH/ESC Guidelines 2023",
        metadata={}
    ),
    ClinicalFact(
        fact_id="F005",
        fact_text="Insulin therapy is not first-line for type 2 diabetes.",
        category="treatment_guidelines",
        source="ADA Guidelines 2024",
        metadata={}
    )
]

# Test fact verification
fact_checker = FactVerificationChecker(knowledge_base)

test_response_correct = LLMResponse(
    response_id="R7",
    query="What is metformin dosing?",
    response_text="Metformin should be started at 500mg once daily with meals and titrated to target dose of 2000mg daily.",
    metadata={}
)

test_response_incorrect = LLMResponse(
    response_id="R8",
    query="What is aspirin dosing for cardiovascular prevention?",
    response_text="Aspirin for cardiovascular prevention should be 1000mg daily.",
    metadata={}
)

print("\nFact Verification Results:\n")

result_correct = fact_checker.verify_response(test_response_correct)
print("Correct Dosage Response:")
print(f"  Hallucination Detected: {result_correct.is_hallucination}")
print(f"  Confidence: {result_correct.confidence:.3f}")
print(f"  Claims Verified: {len([v for v in result_correct.evidence['verifications'] if v['verified']])}/{result_correct.evidence['num_claims']}")

print("\nIncorrect Dosage Response:")
result_incorrect = fact_checker.verify_response(test_response_incorrect)
print(f"  Hallucination Detected: {result_incorrect.is_hallucination}")
print(f"  Confidence: {result_incorrect.confidence:.3f}")
for v in result_incorrect.evidence['verifications']:
    print(f"  Claim '{v['claim']}': {'VERIFIED' if v['verified'] else 'unverified'} (similarity {v['similarity']:.2f})")
    if v['matched_fact']:
        print(f"    matched fact: {v['matched_fact']}")

# Did the checker catch the wrong dose? (computed)
if not result_incorrect.is_hallucination:
    print("\n⚠️  MISSED: the 1000mg aspirin claim was 'verified'. TF-IDF matched the WORDS around the number")
    print("   ('Aspirin for cardiovascular prevention') to the fact above, but never compared the dose itself.")
    print("   Exercise: after finding the best-matching fact, check the claimed dose against that fact's dose range.")
else:
    print("\n✓ The wrong aspirin dose was flagged.")


Fact Verification Results:

Correct Dosage Response:
  Hallucination Detected: False
  Confidence: 1.000
  Claims Verified: 2/2

Incorrect Dosage Response:
  Hallucination Detected: False
  Confidence: 1.000
  Claim '1000mg': VERIFIED (similarity 0.79)
    matched fact: Aspirin for cardiovascular prevention is 81-100mg daily.

⚠️  MISSED: the 1000mg aspirin claim was 'verified'. TF-IDF matched the WORDS around the number
   ('Aspirin for cardiovascular prevention') to the fact above, but never compared the dose itself.
   Exercise: after finding the best-matching fact, check the claimed dose against that fact's dose range.


## Part 4: Entailment-Based Hallucination Detection

### Method: Check Logical Consistency Using NLI

In [7]:
class EntailmentChecker:
    """
    Detects hallucinations by checking if response is entailed by source text.
    Simulated version - in production, use NLI models.
    """

    def __init__(self):
        pass

    def _extract_entities(self, text: str) -> Set[str]:
        """Extract key entities (medications, conditions, numbers)."""
        entities = set()

        # Extract medications (simplified)
        medication_pattern = r'\b(metformin|aspirin|insulin|lisinopril|atorvastatin|levothyroxine)\b'
        medications = re.findall(medication_pattern, text, re.IGNORECASE)
        entities.update([m.lower() for m in medications])

        # Extract numbers
        numbers = re.findall(r'\b\d+(?:\.\d+)?', text)  # no trailing \b, so '500mg' yields '500'
        entities.update(numbers)

        # Extract units
        units = re.findall(r'\b(mg|g|ml|mcg|units?|mmol|mmHg)\b', text, re.IGNORECASE)
        entities.update([u.lower() for u in units])

        return entities

    def check_entailment(self, source_text: str, claim: str) -> Dict:
        """Check if claim is entailed by source text."""
        # Extract entities from both
        source_entities = self._extract_entities(source_text)
        claim_entities = self._extract_entities(claim)

        # Check if claim entities are subset of source entities
        unsupported_entities = claim_entities - source_entities

        # Calculate entailment score
        if claim_entities:
            entailment_score = len(claim_entities & source_entities) / len(claim_entities)
        else:
            entailment_score = 1.0

        is_entailed = len(unsupported_entities) == 0 and entailment_score > 0.7

        return {
            'is_entailed': is_entailed,
            'entailment_score': entailment_score,
            'source_entities': source_entities,
            'claim_entities': claim_entities,
            'unsupported_entities': unsupported_entities
        }

    def detect_hallucination(self, source_text: str,
                            response: LLMResponse) -> HallucinationDetectionResult:
        """Detect hallucination by checking entailment."""
        # Split response into sentences
        sentences = re.split(r'(?<!\d)[.!?]+(?!\d)', response.response_text)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

        # Check each sentence
        entailment_results = []
        for sentence in sentences:
            result = self.check_entailment(source_text, sentence)
            entailment_results.append({
                'sentence': sentence,
                **result
            })

        # Determine if hallucination
        if not entailment_results:
            is_hallucination = False
            confidence = 0.0
        else:
            entailed_count = sum(1 for r in entailment_results if r['is_entailed'])
            entailment_rate = entailed_count / len(entailment_results)

            is_hallucination = entailment_rate < 0.5
            confidence = 1.0 - entailment_rate if is_hallucination else entailment_rate

        return HallucinationDetectionResult(
            response_id=response.response_id,
            is_hallucination=is_hallucination,
            confidence=confidence,
            hallucination_type='unsupported_claim' if is_hallucination else None,
            evidence={
                'entailment_rate': entailment_rate if entailment_results else None,
                'sentence_results': entailment_results
            }
        )

# Test entailment checker
entailment_checker = EntailmentChecker()

source_document = """
Metformin is the first-line pharmacological therapy for type 2 diabetes mellitus.
The starting dose is 500mg once or twice daily with meals. The target dose is
2000mg daily in divided doses. Common side effects include gastrointestinal upset.
Monitor renal function quarterly as metformin is contraindicated in severe renal impairment.
"""

test_response_entailed = LLMResponse(
    response_id="R9",
    query="Summarize metformin prescribing",
    response_text="Metformin 500mg is the starting dose for diabetes. Monitor renal function regularly.",
    metadata={}
)

test_response_not_entailed = LLMResponse(
    response_id="R10",
    query="Summarize metformin prescribing",
    response_text="Metformin 1500mg is the starting dose. It requires liver function monitoring every 2 weeks.",
    metadata={}
)

print("\nEntailment Check Results:\n")

result_entailed = entailment_checker.detect_hallucination(source_document, test_response_entailed)
print("Entailed Response:")
print(f"  Hallucination Detected: {result_entailed.is_hallucination}")
print(f"  Confidence: {result_entailed.confidence:.3f}")
print(f"  Entailment Rate: {result_entailed.evidence['entailment_rate']:.3f}")

print("\nNot Entailed Response:")
result_not_entailed = entailment_checker.detect_hallucination(source_document, test_response_not_entailed)
print(f"  Hallucination Detected: {result_not_entailed.is_hallucination}")
print(f"  Confidence: {result_not_entailed.confidence:.3f}")
print(f"  Entailment Rate: {result_not_entailed.evidence['entailment_rate']:.3f}")
if result_not_entailed.evidence['sentence_results']:
    for sent_result in result_not_entailed.evidence['sentence_results']:
        if not sent_result['is_entailed'] and sent_result['unsupported_entities']:
            print(f"  Unsupported entities: {sent_result['unsupported_entities']}")

if result_not_entailed.is_hallucination and not result_entailed.is_hallucination:
    print("\n✓ The entity check separates the two responses here (it compares numbers/drug names, not meaning).")
else:
    print("\n⚠️  The entity check did NOT separate the two responses. Sentences it accepted as 'supported':")
    for r in result_not_entailed.evidence['sentence_results']:
        if r['is_entailed']:
            print(f"   • \"{r['sentence']}\" — entities {sorted(r['claim_entities'])} all occur somewhere in the source")
    print("   Entity overlap is not meaning: a real NLI model (Part 7) checks whether the source supports the claim.")


Entailment Check Results:

Entailed Response:
  Hallucination Detected: False
  Confidence: 1.000
  Entailment Rate: 1.000

Not Entailed Response:
  Hallucination Detected: False
  Confidence: 0.500
  Entailment Rate: 0.500
  Unsupported entities: {'1500'}

⚠️  The entity check did NOT separate the two responses. Sentences it accepted as 'supported':
   • "It requires liver function monitoring every 2 weeks" — entities ['2'] all occur somewhere in the source
   Entity overlap is not meaning: a real NLI model (Part 7) checks whether the source supports the claim.


## Part 5: Comprehensive Hallucination Detection Pipeline

In [8]:
class HallucinationDetectionPipeline:
    """
    Comprehensive pipeline combining multiple detection methods.
    """

    def __init__(self,
                 consistency_checker: Optional[SelfConsistencyChecker] = None,
                 fact_checker: Optional[FactVerificationChecker] = None,
                 entailment_checker: Optional[EntailmentChecker] = None):
        self.consistency_checker = consistency_checker or SelfConsistencyChecker()
        self.fact_checker = fact_checker
        self.entailment_checker = entailment_checker or EntailmentChecker()

    def detect(self,
              response: LLMResponse,
              alternative_responses: Optional[List[LLMResponse]] = None,
              source_text: Optional[str] = None) -> Dict:
        """Run comprehensive hallucination detection."""
        results = {
            'response_id': response.response_id,
            'query': response.query,
            'checks': {}
        }

        # 1. Consistency check (if alternative responses provided)
        if alternative_responses and len(alternative_responses) > 0:
            all_responses = [response] + alternative_responses
            consistency_result = self.consistency_checker.check_consistency(all_responses)
            results['checks']['consistency'] = {
                'is_hallucination': consistency_result.is_hallucination,
                'confidence': consistency_result.confidence,
                'evidence': consistency_result.evidence
            }

        # 2. Fact verification (if fact checker available)
        if self.fact_checker:
            fact_result = self.fact_checker.verify_response(response)
            results['checks']['fact_verification'] = {
                'is_hallucination': fact_result.is_hallucination,
                'confidence': fact_result.confidence,
                'evidence': fact_result.evidence
            }

        # 3. Entailment check (if source text provided)
        if source_text:
            entailment_result = self.entailment_checker.detect_hallucination(source_text, response)
            results['checks']['entailment'] = {
                'is_hallucination': entailment_result.is_hallucination,
                'confidence': entailment_result.confidence,
                'evidence': entailment_result.evidence
            }

        # Aggregate results
        hallucination_flags = []
        confidences = []

        for check_name, check_result in results['checks'].items():
            hallucination_flags.append(check_result['is_hallucination'])
            confidences.append(check_result['confidence'])

        if hallucination_flags:
            # Majority vote
            hallucination_count = sum(hallucination_flags)
            is_hallucination = hallucination_count > len(hallucination_flags) / 2
            avg_confidence = np.mean(confidences)
        else:
            is_hallucination = False
            avg_confidence = 0.0

        results['overall'] = {
            'is_hallucination': is_hallucination,
            'confidence': avg_confidence,
            'num_checks': len(hallucination_flags),
            'hallucination_count': sum(hallucination_flags) if hallucination_flags else 0
        }

        return results

    def get_recommendation(self, detection_result: Dict) -> str:
        """Get clinical recommendation based on detection results."""
        if not detection_result['checks']:
            return "Insufficient checks performed. Manual review recommended."

        overall = detection_result['overall']

        if overall['is_hallucination']:
            if overall['confidence'] > 0.8:
                return "HIGH RISK: Likely hallucination detected. DO NOT use clinically. Regenerate response."
            elif overall['confidence'] > 0.5:
                return "MODERATE RISK: Possible hallucination. Verify with additional sources before clinical use."
            else:
                return "LOW RISK: Minor inconsistencies detected. Review and verify critical information."
        else:
            if overall['confidence'] > 0.8:
                return "LOW RISK: Response appears reliable. Standard clinical review recommended."
            else:
                return "MODERATE RISK: Response inconsistent. Additional verification recommended."

# Create comprehensive pipeline
pipeline = HallucinationDetectionPipeline(
    consistency_checker=consistency_checker,
    fact_checker=fact_checker,
    entailment_checker=entailment_checker
)

# Test pipeline
print("\n" + "="*80)
print("COMPREHENSIVE HALLUCINATION DETECTION PIPELINE")
print("="*80)

test_response = LLMResponse(
    response_id="R11",
    query="What is metformin dosing for type 2 diabetes?",
    response_text="Metformin should be started at 500mg daily and titrated to 2000mg daily in divided doses.",
    metadata={}
)

alternative_responses = [
    LLMResponse(
        response_id="R12",
        query="What is metformin dosing for type 2 diabetes?",
        response_text="Start metformin at 500mg once or twice daily, titrate to target of 2000mg daily.",
        metadata={}
    ),
    LLMResponse(
        response_id="R13",
        query="What is metformin dosing for type 2 diabetes?",
        response_text="Metformin 500mg is the initial dose. Target dose is 1000-2000mg daily.",
        metadata={}
    )
]

detection_result = pipeline.detect(
    response=test_response,
    alternative_responses=alternative_responses,
    source_text=source_document
)

print(f"\nQuery: {detection_result['query']}")
print(f"Response ID: {detection_result['response_id']}")
print("\nDetection Results:")

for check_name, check_result in detection_result['checks'].items():
    print(f"\n{check_name.upper()}:")
    print(f"  Hallucination: {check_result['is_hallucination']}")
    print(f"  Confidence: {check_result['confidence']:.3f}")

print(f"\nOVERALL ASSESSMENT:")
print(f"  Hallucination Detected: {detection_result['overall']['is_hallucination']}")
print(f"  Average Confidence: {detection_result['overall']['confidence']:.3f}")
print(f"  Checks Performed: {detection_result['overall']['num_checks']}")

recommendation = pipeline.get_recommendation(detection_result)
print(f"\nCLINICAL RECOMMENDATION:\n  {recommendation}")

flagged = [k for k, v in detection_result['checks'].items() if v['is_hallucination']]
if flagged:
    print(f"\nNote: R11 is a correct answer (and agrees with R12/R13), yet {', '.join(flagged)} flagged it — a false alarm")
    print(f"      {'outvoted by the other checks' if not detection_result['overall']['is_hallucination'] else 'that the majority vote did NOT correct'}.")


COMPREHENSIVE HALLUCINATION DETECTION PIPELINE

Query: What is metformin dosing for type 2 diabetes?
Response ID: R11

Detection Results:

CONSISTENCY:
  Hallucination: True
  Confidence: 0.775

FACT_VERIFICATION:
  Hallucination: False
  Confidence: 1.000

ENTAILMENT:
  Hallucination: False
  Confidence: 1.000

OVERALL ASSESSMENT:
  Hallucination Detected: False
  Average Confidence: 0.925
  Checks Performed: 3

CLINICAL RECOMMENDATION:
  LOW RISK: Response appears reliable. Standard clinical review recommended.

Note: R11 is a correct answer (and agrees with R12/R13), yet consistency flagged it — a false alarm
      outvoted by the other checks.


## Part 6: Evaluation and Visualization

In [9]:
# Create test dataset with ground truth
test_dataset = [
    {
        'response': LLMResponse(
            response_id="T1",
            query="Aspirin dosing for CVD prevention?",
            response_text="Aspirin 81-100mg daily for cardiovascular prevention.",
            metadata={}
        ),
        'alternatives': [
            LLMResponse("T1a", "Aspirin dosing for CVD prevention?", "Aspirin 75-100mg daily.", {}),
            LLMResponse("T1b", "Aspirin dosing for CVD prevention?", "Low-dose aspirin 81mg daily.", {})
        ],
        'source': "Aspirin 81-100mg daily is recommended for cardiovascular prevention in high-risk patients.",
        'ground_truth_hallucination': False
    },
    {
        'response': LLMResponse(
            response_id="T2",
            query="Aspirin dosing for CVD prevention?",
            response_text="Aspirin 1000mg daily for cardiovascular prevention.",
            metadata={}
        ),
        'alternatives': [
            LLMResponse("T2a", "Aspirin dosing for CVD prevention?", "Aspirin 975mg daily.", {}),
            LLMResponse("T2b", "Aspirin dosing for CVD prevention?", "High-dose aspirin 1200mg.", {})
        ],
        'source': "Aspirin 81-100mg daily is recommended for cardiovascular prevention in high-risk patients.",
        'ground_truth_hallucination': True
    },
    {
        'response': LLMResponse(
            response_id="T3",
            query="Lisinopril starting dose?",
            response_text="Lisinopril 10mg daily is the starting dose for hypertension.",
            metadata={}
        ),
        'alternatives': [
            LLMResponse("T3a", "Lisinopril starting dose?", "Start lisinopril at 10mg once daily.", {}),
            LLMResponse("T3b", "Lisinopril starting dose?", "Initial lisinopril dose is 10mg.", {})
        ],
        'source': "Lisinopril starting dose for hypertension is typically 10mg once daily.",
        'ground_truth_hallucination': False
    },
    {
        'response': LLMResponse(
            response_id="T4",
            query="Metformin starting dose?",
            response_text="Metformin 2000mg daily is the starting dose.",
            metadata={}
        ),
        'alternatives': [
            LLMResponse("T4a", "Metformin starting dose?", "Start metformin at 500mg daily.", {}),
            LLMResponse("T4b", "Metformin starting dose?", "Metformin 1000mg initially.", {})
        ],
        'source': "Metformin starting dose is 500mg once or twice daily with meals.",
        'ground_truth_hallucination': True
    }
]

# Run detection on test dataset
results = []
check_flags = {}
for test_case in test_dataset:
    detection = pipeline.detect(
        response=test_case['response'],
        alternative_responses=test_case['alternatives'],
        source_text=test_case['source']
    )
    check_flags[test_case['response'].response_id] = {k: bool(v['is_hallucination']) for k, v in detection['checks'].items()}

    results.append({
        'response_id': test_case['response'].response_id,
        'predicted_hallucination': detection['overall']['is_hallucination'],
        'confidence': detection['overall']['confidence'],
        'ground_truth': test_case['ground_truth_hallucination']
    })

results_df = pd.DataFrame(results)

# Calculate metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_true = results_df['ground_truth'].values
y_pred = results_df['predicted_hallucination'].values

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("\n" + "="*80)
print("HALLUCINATION DETECTION PERFORMANCE")
print("="*80)
print(f"\nAccuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 Score:  {f1:.3f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Hallucination', 'Hallucination'],
            yticklabels=['Not Hallucination', 'Hallucination'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Confidence distribution
results_df_true_hallu = results_df[results_df['ground_truth'] == True]
results_df_true_valid = results_df[results_df['ground_truth'] == False]

if len(results_df_true_hallu) > 0:
    axes[1].hist(results_df_true_hallu['confidence'], bins=5, alpha=0.7, label='True Hallucinations', color='red')
if len(results_df_true_valid) > 0:
    axes[1].hist(results_df_true_valid['confidence'], bins=5, alpha=0.7, label='Valid Responses', color='green')

axes[1].set_xlabel('Detection Confidence')
axes[1].set_ylabel('Count')
axes[1].set_title('Confidence Distribution by Ground Truth')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nDetailed Results:")
print(results_df.to_string(index=False))

print("\nPer-check flags (True = that check called it a hallucination):")
for rid, flags in check_flags.items():
    print(f"  {rid}: {flags}")
print("\nEach check on its own:")
for check in next(iter(check_flags.values())):
    preds = np.array([check_flags[rid][check] for rid in results_df['response_id']])
    print(f"  {check:18s}: {np.mean(preds == y_true):.0%} correct, flagged {preds.sum()}/{len(preds)}")
    if preds.all() or not preds.any():
        print(f"    ⚠️  {check} flagged {'every' if preds.all() else 'no'} case, so it carries no information here")

# Honest verdict, computed from the results above
n_true = int(y_true.sum())
missed = results_df[results_df['ground_truth'] & ~results_df['predicted_hallucination']]['response_id'].tolist()
false_alarms = results_df[~results_df['ground_truth'] & results_df['predicted_hallucination']]['response_id'].tolist()
print(f"\nCaught {n_true - len(missed)} of {n_true} true hallucinations; {len(false_alarms)} false alarm(s).")
if missed:
    print(f"⚠️  MISSED: {', '.join(missed)} — despite 'confidence' up to {results_df['confidence'].max():.2f}.")
    print("   High confidence from a detector is not evidence that it is right.")
if false_alarms:
    print(f"⚠️  False alarms: {', '.join(false_alarms)}")
print(f"   (Only {len(results_df)} test cases: a demo of the workflow, not a validation of these detectors.)")


HALLUCINATION DETECTION PERFORMANCE

Accuracy:  1.000
Precision: 1.000
Recall:    1.000
F1 Score:  1.000



Detailed Results:
response_id  predicted_hallucination  confidence  ground_truth
         T1                    False    0.914815         False
         T2                     True    0.941769          True
         T3                    False    0.902694         False
         T4                     True    0.955497          True

Per-check flags (True = that check called it a hallucination):
  T1: {'consistency': True, 'fact_verification': False, 'entailment': False}
  T2: {'consistency': True, 'fact_verification': False, 'entailment': True}
  T3: {'consistency': True, 'fact_verification': False, 'entailment': False}
  T4: {'consistency': True, 'fact_verification': False, 'entailment': True}

Each check on its own:
  consistency       : 50% correct, flagged 4/4
    ⚠️  consistency flagged every case, so it carries no information here
  fact_verification : 50% correct, flagged 0/4
    ⚠️  fact_verification flagged no case, so it carries no information here
  entailment        : 100% 

## Part 6b: Uncertainty Elicitation and Calibration

**Uncertainty elicitation** asks the model to state how confident it is. That is only useful if stated confidence **predicts accuracy**: answers given with 90% confidence should be right about 90% of the time. We test this on the 12 answers in the test set above (4 main responses + 8 alternatives).

- **Correctness** is computed: an answer is correct if every dose it states is in the accepted set for that question (taken from the source text).
- **Stated confidence** is *simulated*: the values below were chosen by us to mimic the overconfidence commonly reported for LLMs. They are synthetic, so the result illustrates the analysis, not any real model. With a key, use the prompt shown to collect real values.

Metrics: **accuracy vs. mean confidence** (over/under-confidence), **ECE**, **Brier score** (mean squared error of confidence vs. correct, lower is better), and **AUROC** (does higher confidence rank correct answers above wrong ones? 0.5 = no better than chance).

In [10]:
UNCERTAINTY_ELICITATION_PROMPT = """{query}
Answer in one sentence. Then on a new line write 'Confidence: N%' where N is the probability (0-100)
that your answer is correct. If you are unsure, say so and give a low number."""
print("Uncertainty-elicitation prompt template:\n")
print(UNCERTAINTY_ELICITATION_PROMPT.format(query="Metformin starting dose?"))

# SIMULATED stated confidences (synthetic, chosen to mimic LLM overconfidence)
STATED_CONFIDENCE = {'T1': 0.92, 'T1a': 0.85, 'T1b': 0.90, 'T2': 0.88, 'T2a': 0.80, 'T2b': 0.75,
                     'T3': 0.95, 'T3a': 0.93, 'T3b': 0.90, 'T4': 0.90, 'T4a': 0.80, 'T4b': 0.85}
# Accepted doses (mg) per question, from the source texts in the test set
ACCEPTED_DOSES = {'Aspirin dosing for CVD prevention?': {75, 81, 100},
                  'Lisinopril starting dose?': {10},
                  'Metformin starting dose?': {500}}

calib_rows = []
for tc in test_dataset:
    for r in [tc['response']] + tc['alternatives']:
        doses = {int(x) for x in re.findall(r'\d+', r.response_text)}
        calib_rows.append({'id': r.response_id, 'answer': r.response_text[:45],
                           'stated confidence': STATED_CONFIDENCE[r.response_id],
                           'correct': bool(doses) and doses <= ACCEPTED_DOSES[r.query]})
calib_df = pd.DataFrame(calib_rows)
print("\n" + calib_df.to_string(index=False))

from sklearn.metrics import roc_auc_score, brier_score_loss
conf = calib_df['stated confidence'].values
correct = calib_df['correct'].values.astype(float)
bins = np.linspace(0, 1, 11)
ece = sum(((conf > lo) & (conf <= hi)).mean() * abs(correct[(conf > lo) & (conf <= hi)].mean() - conf[(conf > lo) & (conf <= hi)].mean())
          for lo, hi in zip(bins[:-1], bins[1:]) if ((conf > lo) & (conf <= hi)).any())
auroc = roc_auc_score(correct, conf)
brier = brier_score_loss(correct, conf)
gap = conf.mean() - correct.mean()
verdict = ('over-confident' if gap > 0.05 else 'under-confident' if gap < -0.05 else 'roughly calibrated on average')
print(f"\nAccuracy {correct.mean():.0%} vs mean stated confidence {conf.mean():.0%} → {verdict} "
      f"(gap {gap * 100:+.0f} percentage points)")
print(f"ECE {ece:.2f} | Brier {brier:.3f} | AUROC {auroc:.2f}")
print(f"Mean confidence when correct: {conf[correct == 1].mean():.2f}; when wrong: {conf[correct == 0].mean():.2f}")
wrong_conf = calib_df[~calib_df['correct'] & (calib_df['stated confidence'] >= 0.85)]
if len(wrong_conf):
    print(f"⚠️ {len(wrong_conf)} wrong answer(s) stated with ≥85% confidence: {', '.join(wrong_conf['id'])} "
          f"(e.g. '{wrong_conf.iloc[0]['answer']}').")
if auroc < 0.6:
    print("Stated confidence barely separates right from wrong answers here: do not use it alone as a filter.")
elif auroc < 0.8:
    print("Stated confidence carries some signal, but not enough to replace verification against a source.")
else:
    print("Stated confidence ranks right above wrong answers fairly well in this sample (AUROC), ...")
if ece > 0.10:
    for lo, hi in [(0.0, 0.90), (0.90, 1.01)]:
        m = (conf >= lo) & (conf < hi)
        if m.any():
            print(f"   answers stated at {conf[m].min():.0%}–{conf[m].max():.0%} confidence were right {correct[m].mean():.0%} of the time (n={m.sum()})")
    print(f"...so the numbers themselves are not trustworthy probabilities (ECE {ece:.2f}).")
    print("Ranking (discrimination) and honesty of the numbers (calibration) are different properties; check both.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
mids, accs, ns = [], [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (conf > lo) & (conf <= hi)
    if m.any():
        mids.append(conf[m].mean()); accs.append(correct[m].mean()); ns.append(m.sum())
ax[0].scatter(mids, accs, s=[40 * k for k in ns], zorder=3, label='bins (size = n)')
ax[0].plot(mids, accs, 'o-', alpha=0.6)
ax[0].set_xlabel('Stated confidence'); ax[0].set_ylabel('Observed accuracy'); ax[0].set_xlim(0, 1); ax[0].set_ylim(-0.05, 1.05)
ax[0].set_title(f'Reliability (ECE {ece:.2f}, n={len(conf)}, simulated confidences)'); ax[0].legend(fontsize=8)
ax[1].hist([conf[correct == 1], conf[correct == 0]], bins=np.linspace(0.7, 1.0, 7), label=['correct', 'wrong'], color=['green', 'red'], alpha=0.7)
ax[1].set_xlabel('Stated confidence'); ax[1].set_ylabel('Count'); ax[1].set_title(f'Confidence by correctness (AUROC {auroc:.2f})'); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"\nOnly {len(conf)} answers: one answer moves these metrics a lot. The method, not the numbers, is the lesson.")

Uncertainty-elicitation prompt template:

Metformin starting dose?
Answer in one sentence. Then on a new line write 'Confidence: N%' where N is the probability (0-100)
that your answer is correct. If you are unsure, say so and give a low number.

 id                                        answer  stated confidence  correct
 T1 Aspirin 81-100mg daily for cardiovascular pre               0.92     True
T1a                       Aspirin 75-100mg daily.               0.85     True
T1b                  Low-dose aspirin 81mg daily.               0.90     True
 T2 Aspirin 1000mg daily for cardiovascular preve               0.88    False
T2a                          Aspirin 975mg daily.               0.80    False
T2b                     High-dose aspirin 1200mg.               0.75    False
 T3 Lisinopril 10mg daily is the starting dose fo               0.95     True
T3a          Start lisinopril at 10mg once daily.               0.93     True
T3b              Initial lisinopril dose is 10mg.  


Only 12 answers: one answer moves these metrics a lot. The method, not the numbers, is the lesson.


## Part 7: Production Implementation Example

In [11]:
production_example = '''
# PRODUCTION HALLUCINATION DETECTION
# Using real LLM APIs and NLI models

import os
from openai import OpenAI  # openai>=1.0
from transformers import pipeline

# Setup
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))

# Load NLI model for entailment checking
nli_model = pipeline("text-classification", model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli")

# === SELF-CONSISTENCY CHECK ===
def generate_multiple_responses(query: str, n: int = 5) -> List[str]:
    """Generate multiple responses with temperature sampling."""
    responses = []
    for _ in range(n):
        response = client.chat.completions.create(
            model="~anthropic/claude-haiku-latest",
            messages=[{"role": "user", "content": query}],
            temperature=0.7,  # Higher temperature for diversity
            max_tokens=200
        )
        responses.append(response.choices[0].message.content)
    return responses

def check_self_consistency(responses: List[str]) -> float:
    """Calculate self-consistency score."""
    # Use NLI to check if responses entail each other
    consistency_scores = []
    for i in range(len(responses)):
        for j in range(i+1, len(responses)):
            result = nli_model(f"{responses[i]} [SEP] {responses[j]}")
            # Score based on entailment/neutral vs contradiction
            label = result[0]['label'].upper()  # this model returns lowercase labels
            if label == 'ENTAILMENT':
                consistency_scores.append(1.0)
            elif label == 'NEUTRAL':
                consistency_scores.append(0.5)
            else:  # CONTRADICTION
                consistency_scores.append(0.0)

    return sum(consistency_scores) / len(consistency_scores) if consistency_scores else 1.0

# === SELF-VERIFICATION ===
def self_verify(query: str, response: str) -> Dict:
    """Ask LLM to critique its own response."""
    verification_prompt = f"""You are a medical fact-checker. Review this response for accuracy.

Query: {query}
Response: {response}

Identify any potentially incorrect information. List specific concerns.
Format: If accurate, respond "VERIFIED". If concerns, list them."""

    verification = client.chat.completions.create(
        model="~anthropic/claude-haiku-latest",
        messages=[{"role": "user", "content": verification_prompt}],
        temperature=0.1,
        max_tokens=300
    )

    result = verification.choices[0].message.content
    is_verified = "VERIFIED" in result.upper()

    return {
        'is_verified': is_verified,
        'verification_text': result
    }

# === ENTAILMENT CHECK ===
def check_entailment(source: str, claim: str) -> float:
    """Check if claim is entailed by source using NLI model."""
    result = nli_model(f"{source} [SEP] {claim}")

    if result[0]['label'].upper() == 'ENTAILMENT':
        return result[0]['score']
    else:
        return 0.0

# === COMPREHENSIVE DETECTION ===
def detect_hallucination(query: str, response: str, source: str = None) -> Dict:
    """Run comprehensive hallucination detection."""
    results = {'is_hallucination': False, 'confidence': 0.0, 'checks': {}}

    # 1. Self-consistency
    alt_responses = generate_multiple_responses(query, n=3)
    all_responses = [response] + alt_responses
    consistency_score = check_self_consistency(all_responses)
    results['checks']['consistency'] = {
        'score': consistency_score,
        'is_hallucination': consistency_score < 0.6
    }

    # 2. Self-verification
    verification = self_verify(query, response)
    results['checks']['self_verification'] = {
        'is_verified': verification['is_verified'],
        'is_hallucination': not verification['is_verified']
    }

    # 3. Entailment (if source provided)
    if source:
        entailment_score = check_entailment(source, response)
        results['checks']['entailment'] = {
            'score': entailment_score,
            'is_hallucination': entailment_score < 0.5
        }

    # Aggregate
    hallucination_flags = [check['is_hallucination'] for check in results['checks'].values()]
    hallucination_count = sum(hallucination_flags)
    results['is_hallucination'] = hallucination_count >= 2  # Majority vote
    results['confidence'] = hallucination_count / len(hallucination_flags)

    return results

# === USAGE ===
query = "What is the dosing for metformin?"
response = "Metformin 500mg twice daily is the starting dose."
source = "Metformin starting dose is 500mg once or twice daily with meals."

result = detect_hallucination(query, response, source)

if result['is_hallucination']:
    print(f"WARNING: Likely hallucination (confidence: {result['confidence']:.2f})")
    print("Do not use for clinical decisions. Regenerate response.")
else:
    print(f"Response appears valid (confidence: {1-result['confidence']:.2f})")

# Cost estimate:
# Per query: ~5 API calls
# ~2500 tokens total
# Cost: ~$0.0007 per detection
# 1000 detections: ~$0.70
'''

print("\nProduction Hallucination Detection Example:")
print(production_example)


Production Hallucination Detection Example:

# PRODUCTION HALLUCINATION DETECTION
# Using real LLM APIs and NLI models

import os
from openai import OpenAI  # openai>=1.0
from transformers import pipeline

# Setup
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))

# Load NLI model for entailment checking
nli_model = pipeline("text-classification", model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli")

# === SELF-CONSISTENCY CHECK ===
def generate_multiple_responses(query: str, n: int = 5) -> List[str]:
    """Generate multiple responses with temperature sampling."""
    responses = []
    for _ in range(n):
        response = client.chat.completions.create(
            model="~anthropic/claude-haiku-latest",
            messages=[{"role": "user", "content": query}],
            temperature=0.7,  # Higher temperature for diversity
            max_tokens=200
        )
        responses.append(response.choices[0].message.content)
    return

## Key Takeaways

### Types of Clinical Hallucinations
1. **Factual Errors**: Incorrect dosages, contraindications, lab values
2. **Temporal Errors**: Wrong timelines, follow-up intervals
3. **Logical Inconsistencies**: Self-contradictory statements
4. **Source Fabrication**: Citing non-existent guidelines

### Detection Methods

*Our simplified (TF-IDF / regex) detectors each failed at least one of their own demos above — compare the computed results. The points below describe what well-calibrated versions aim for.*

1. **Self-Consistency**:
   - Generate 3-5 responses with temperature sampling
   - Check pairwise similarity or entailment
   - Thresholds must be calibrated: with TF-IDF word overlap, even agreeing paraphrases can fall below 0.6

2. **Fact Verification**:
   - Compare numerical claims against knowledge base
   - Check medication names, dosages, lab ranges
   - Similarity threshold: 0.5-0.7
   - Must compare the values themselves: text similarity alone can 'verify' a wrong dose (Part 3)

3. **Entailment Checking**:
   - Use NLI models to verify claims
   - Check if response is supported by source
   - Detects unsupported inferences
   - Requires source documents

4. **Self-Verification**:
   - Ask LLM to critique its own output
   - Effective for obvious errors
   - Low overhead (1 additional API call)

### Best Practices
1. **Multi-Method Approach**: Combine 2-3 detection methods
2. **Majority Voting**: Flag hallucination if ≥2 methods agree
3. **Confidence Thresholds**:
   - High confidence (>0.8): Block response
   - Medium (0.5-0.8): Flag for review
   - Low (<0.5): Standard review

4. **Production Workflow**:
   - Run fast checks first (self-verification)
   - Use expensive checks (multi-generation) for high-risk queries
   - Always require human review for critical decisions

### Performance Targets
- **Precision**: >0.8 (minimize false alarms)
- **Recall**: >0.7 (catch most hallucinations)
- **Latency**: <3 seconds for real-time use
- **Cost**: <$0.001 per detection

### Common Pitfalls
- Over-reliance on single detection method
- Not calibrating thresholds for domain
- Ignoring low-confidence hallucinations
- No human-in-the-loop for high-risk scenarios
- Not updating knowledge base regularly

### Critical for Safety
- Hallucination detection is REQUIRED for clinical LLM deployment
- No system is 100% accurate - always include human review
- Update detection systems as LLMs evolve
- Monitor false negative rate carefully (missed hallucinations)

## Exercises

### Exercise 1: Improve Consistency Checking
Implement semantic clustering of responses to identify majority vs outlier answers. Use K-means or DBSCAN on response embeddings.

### Exercise 2: Build Domain-Specific Knowledge Base
Create a comprehensive knowledge base from clinical guidelines (NICE, AHA/ACC). Implement efficient indexing with FAISS or Elasticsearch.

### Exercise 3: Implement Chain-of-Verification
Build a system where LLM:
1. Generates initial response
2. Lists verifiable claims
3. Checks each claim independently
4. Revises response based on verification

### Exercise 4: Calibrate Detection Thresholds
Use labeled dataset to optimize similarity and consistency thresholds for maximum F1 score. Plot ROC curves.

### Exercise 5: Production NLI Integration
Integrate real NLI model (DeBERTa-v3-MNLI) for entailment checking. Compare performance vs. similarity-based approach.

### Exercise 6: Multi-Model Consensus
Query 3 different LLMs (Claude, GPT-4, Gemini) and flag disagreements. Measure improvement in hallucination detection.

### Bonus Challenge: Adversarial Testing
Create synthetic hallucinations of different types. Test detection system's ability to catch:
- Off-by-one errors (500mg vs 5000mg)
- Subtle contradictions
- Source fabrications
- Temporal errors